# Caritas Westminster Spatial Analysis and Analytical Dashboard

## Research Question

To what extent are social action initiatives aligned with patterns of socioeconomic deprivation across the Diocese of Westminster?

This notebook contains the spatial-analysis, interactive-mapping and analytical-dashboard workflow developed alongside the main Business Analysis notebook.

The spatial workflow uses a narrower analytical population than the main descriptive analysis. The full 2025 APFR dataset contains 605 social-action records, while the mapping and dashboard analysis uses named project records that can be matched reliably to mapped parish geography.

Recorded APFR social-action activity is used as the observable measure of provision. Missing or unmatched records are not treated as zero activity, and the outputs are interpreted as exploratory evidence of geographical alignment rather than measures of project impact or confirmed under-provision.

## 1. Setup and File Configuration

The required Python libraries are loaded for data processing, spatial analysis, interactive mapping and dashboard generation. Input and output paths are then defined for the APFR, church-location and IMD datasets together with the generated HTML outputs.

The workflow produces interactive maps, dashboard graphics, validation files and a connected homepage.

In [ ]:
from pathlib import Path
import os
import re
import html
import json
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
import matplotlib.pyplot as plt

from branca.colormap import linear
from folium.plugins import HeatMap, Fullscreen
from IPython.display import display

BASE_DIR = Path.cwd()

if not (BASE_DIR / "Data").exists():
    candidate = BASE_DIR / "Caritas-Westminster Dissertation"
    if (candidate / "Data").exists():
        BASE_DIR = candidate

DATA_DIR = BASE_DIR / "Data"

if not DATA_DIR.exists():
    raise FileNotFoundError(
        "Data folder not found. Place the supplied datasets in a folder named 'Data'."
    )


def find_data_file(*filenames):
    for filename in filenames:
        path = DATA_DIR / filename
        if path.exists():
            return path
    return None


afr_file = find_data_file(
    "afr_2025.xlsx"
)

churches_file = find_data_file(
    "rcdow_churches.csv"
)

imd_file = find_data_file(
    "File_7_IoD2025_All_Ranks_Scores_Deciles_Population_Denominators.csv",
    "imd_2025.csv"
)

# Optional map icon
cross_icon = BASE_DIR / "cross.png"

if not cross_icon.exists():
    cross_icon = None


required_files = {
    "APFR 2025 workbook": afr_file,
    "Church-location CSV": churches_file,
    "IMD 2025 CSV": imd_file
}

missing_files = [
    name
    for name, path in required_files.items()
    if path is None or not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "These files could not be found:\n"
        + "\n".join(missing_files)
    )


output_folder = BASE_DIR / "caritas_outputs_2025"

output_folder.mkdir(
    parents=True,
    exist_ok=True
)


## 2. APFR and Church-Location Data Preparation

The 2025 APFR social-action data and church-location data are cleaned before geographical matching. Text fields are standardised, numeric fields are converted where possible, and missing values remain missing rather than being interpreted as zero.

A stable record identifier is assigned to each APFR row to preserve record-level traceability. Parish-location keys are also created to support matching between the APFR and church datasets.

The APFR data are separated into related analytical populations. The complete dataset is retained as the source data, records with a project name form the identifiable-project subset, and only named records that can be matched reliably to mapped parish geography are carried forward into the spatial analysis.

In [ ]:
def clean_text(series):
    return (
        series.astype("string")
        .str.strip()
        .replace(
            {
                "": pd.NA,
                "nan": pd.NA,
                "NaN": pd.NA,
                "NAN": pd.NA,
                "None": pd.NA,
                "none": pd.NA,
                "NA": pd.NA,
                "N/A": pd.NA,
                "Other: NA": pd.NA
            }
        )
    )

def normalise_key_value(value):
    if pd.isna(value):
        return ""

    value = str(value).strip().lower()
    value = value.replace("&", " and ")
    value = re.sub(
        r"[^a-z0-9]+",
        " ",
        value
    )
    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()

def create_location_key(frame):
    return (
        frame["episcopal_area"].map(
            normalise_key_value
        )
        + "|"
        + frame["deanery"].map(
            normalise_key_value
        )
        + "|"
        + frame["parish"].map(
            normalise_key_value
        )
    )

def split_category_tokens(value):
    if pd.isna(value):
        return []

    return [
        token.strip().lower()
        for token in str(value).split(";")
        if token.strip()
    ]

def sum_or_missing(series):
    return series.sum(
        min_count=1
    )

def combine_unique_text(
    series,
    limit=15
):
    values = list(
        dict.fromkeys(
            str(value).strip()
            for value in series.dropna()
            if str(value).strip()
        )
    )

    if len(values) > limit:
        return (
            "; ".join(values[:limit])
            + f"; +{len(values) - limit} more"
        )

    return "; ".join(values)

def safe_html(
    value,
    fallback=""
):
    if pd.isna(value):
        return fallback

    return html.escape(
        str(value)
    )

df_churches = pd.read_csv(
    churches_file
)

df_projects_raw = pd.read_excel(
    afr_file,
    sheet_name="social action"
)

church_text_columns = [
    "episcopal_area",
    "deanery",
    "parish",
    "church",
    "postcode",
    "admin_district"
]

for column in church_text_columns:
    if column in df_churches.columns:
        df_churches[column] = clean_text(
            df_churches[column]
        )

project_text_columns = [
    "episcopal_area",
    "deanery",
    "parish",
    "project",
    "description",
    "category",
    "frequency"
]

for column in project_text_columns:
    if column in df_projects_raw.columns:
        df_projects_raw[column] = clean_text(
            df_projects_raw[column]
        )

df_churches["longitude"] = pd.to_numeric(
    df_churches["longitude"],
    errors="coerce"
)

df_churches["latitude"] = pd.to_numeric(
    df_churches["latitude"],
    errors="coerce"
)

df_projects_raw["people_supported"] = pd.to_numeric(
    df_projects_raw["people_supported"],
    errors="coerce"
)

df_projects_raw["num_sa_volunteers"] = pd.to_numeric(
    df_projects_raw["num_sa_volunteers"],
    errors="coerce"
)

northfields_flag = (
    df_projects_raw["parish"].eq("Northfields")
    & df_projects_raw["project"].str.strip().str.lower().eq("svp")
    & df_projects_raw["people_supported"].eq(46083)
)

warm_hub_flag = (
    df_projects_raw["parish"].eq("West Drayton and Yiewsley")
    & df_projects_raw["project"].str.strip().str.lower().eq("warm hub")
    & df_projects_raw["people_supported"].eq(41974)
)

df_projects_raw["people_supported_flagged"] = (
    northfields_flag | warm_hub_flag
)

df_projects_raw["people_supported_for_analysis"] = (
    df_projects_raw["people_supported"]
    .mask(df_projects_raw["people_supported_flagged"])
)
df_projects_raw["record_id"] = np.arange(
    1,
    len(df_projects_raw) + 1
)

df_projects_raw["category_tokens"] = (
    df_projects_raw["category"]
    .apply(split_category_tokens)
)

df_projects_raw["project_search_text"] = (
    df_projects_raw["project"]
    .fillna("")
    .astype(str)
    + " "
    + df_projects_raw["description"]
    .fillna("")
    .astype(str)
    + " "
    + df_projects_raw["category"]
    .fillna("")
    .astype(str)
).str.lower()

df_churches = (
    df_churches
    .dropna(
        subset=[
            "longitude",
            "latitude"
        ]
    )
    .copy()
)

df_churches["location_key"] = create_location_key(
    df_churches
)

df_projects_raw["location_key"] = create_location_key(
    df_projects_raw
)

parish_locations = (
    df_churches
    .groupby(
        "location_key",
        as_index=False
    )
    .agg(
        episcopal_area=(
            "episcopal_area",
            "first"
        ),
        deanery=(
            "deanery",
            "first"
        ),
        parish=(
            "parish",
            "first"
        ),
        longitude=(
            "longitude",
            "median"
        ),
        latitude=(
            "latitude",
            "median"
        ),
        church_count=(
            "church",
            "nunique"
        )
    )
)

named_projects = (
    df_projects_raw[
        df_projects_raw["project"].notna()
    ]
    .copy()
)

named_projects_geocoded = (
    named_projects
    .merge(
        parish_locations[
            [
                "location_key",
                "longitude",
                "latitude",
                "church_count"
            ]
        ],
        on="location_key",
        how="left",
        validate="many_to_one"
    )
)

unmatched_named_projects = (
    named_projects_geocoded[
        named_projects_geocoded["longitude"].isna()
        | named_projects_geocoded["latitude"].isna()
    ][
        [
            "episcopal_area",
            "deanery",
            "parish",
            "project"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "episcopal_area",
            "deanery",
            "parish",
            "project"
        ]
    )
)

matched_named_projects = (
    named_projects_geocoded
    .dropna(
        subset=[
            "longitude",
            "latitude"
        ]
    )
    .copy()
)

if matched_named_projects[
    "record_id"
].duplicated().any():
    raise ValueError(
        "Project records were duplicated during "
        "the parish-location merge."
    )

if not unmatched_named_projects.empty:
    unmatched_named_projects.to_csv(
        output_folder
        / "unmatched_named_projects.csv",
        index=False
    )

print("Raw APFR rows:", len(df_projects_raw))
print("Named project records:", len(named_projects))
print(
    "Named project records matched to parish geography:",
    len(matched_named_projects)
)
print(
    "Named project records without a parish match:",
    len(unmatched_named_projects)
)
print(
    "Unique geocoded parishes:",
    parish_locations["location_key"].nunique()
)

## 3. IMD 2025 and LSOA Boundaries

The English Indices of Deprivation 2025 provide the socioeconomic measures used in the spatial analysis. Overall IMD and the seven deprivation domains are retained: income, employment, education, health, crime, barriers to housing and services, and living environment.

The IMD data are restricted to the local authorities represented in the study area and linked to official 2021 Lower-layer Super Output Area boundaries using LSOA codes.

Boundary matching is checked before the analysis continues so that deprivation values can be represented consistently across the study geography.

In [ ]:
df_imd = pd.read_csv(
    imd_file
)

df_imd = df_imd.rename(
    columns={
        "LSOA code (2021)": "lsoa_code",
        "Local Authority District name (2024)": "LADnm",
        "Index of Multiple Deprivation (IMD) Score": "IMDScore",
        "Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)": "IMDDecile",
        "Income Score (rate)": "IncomeScore",
        "Employment Score (rate)": "EmpScore",
        "Education, Skills and Training Score": "EduScore",
        "Health Deprivation and Disability Score": "HealthScore",
        "Crime Score": "CrimeScore",
        "Barriers to Housing and Services Score": "HousingScore",
        "Living Environment Score": "EnvScore"
    }
)

required_imd_columns = [
    "lsoa_code",
    "LADnm",
    "IMDScore",
    "IMDDecile",
    "IncomeScore",
    "EmpScore",
    "EduScore",
    "HealthScore",
    "CrimeScore",
    "HousingScore",
    "EnvScore"
]

missing_imd_columns = [
    column
    for column in required_imd_columns
    if column not in df_imd.columns
]

if missing_imd_columns:
    raise KeyError(
        "These IMD columns were not found:\n"
        + "\n".join(missing_imd_columns)
    )

study_lads = [
    "Barnet",
    "Brent",
    "Broxbourne",
    "Camden",
    "City of London",
    "Dacorum",
    "Ealing",
    "East Hertfordshire",
    "Enfield",
    "Hackney",
    "Hammersmith and Fulham",
    "Haringey",
    "Harrow",
    "Hertsmere",
    "Hillingdon",
    "Hounslow",
    "Islington",
    "Kensington and Chelsea",
    "North Hertfordshire",
    "Richmond upon Thames",
    "Spelthorne",
    "Stevenage",
    "St Albans",
    "Three Rivers",
    "Tower Hamlets",
    "Watford",
    "Welwyn Hatfield",
    "Westminster"
]

df_imd = (
    df_imd[
        df_imd["LADnm"].isin(
            study_lads
        )
    ]
    .copy()
)

df_imd["lsoa_code"] = clean_text(
    df_imd["lsoa_code"]
)

boundary_cache = (
    output_folder
    / "lsoa_2021_study_area.geojson"
)

if boundary_cache.exists():
    gdf_boundaries = gpd.read_file(
        boundary_cache
    )

    print(
        "Using cached official 2021 LSOA boundaries:",
        boundary_cache
    )

else:
    boundary_service_url = (
        "https://services1.arcgis.com/"
        "ESMARspQHYMw9BZ9/ArcGIS/rest/services/"
        "Lower_layer_Super_Output_Areas_December_2021_"
        "Boundaries_EW_BGC_V5/FeatureServer/0/query"
    )

    all_features = []
    result_offset = 0
    record_count = 2000

    while True:
        parameters = {
            "where": "1=1",
            "outFields": (
                "FID,LSOA21CD,LSOA21NM"
            ),
            "returnGeometry": "true",
            "outSR": "4326",
            "f": "geojson",
            "orderByFields": "FID",
            "resultOffset": result_offset,
            "resultRecordCount": record_count
        }

        response = requests.get(
            boundary_service_url,
            params=parameters,
            timeout=120
        )

        response.raise_for_status()
        payload = response.json()

        features = payload.get(
            "features",
            []
        )

        if not features:
            break

        all_features.extend(
            features
        )

        print(
            "Downloaded boundary features:",
            len(all_features)
        )

        if len(features) < record_count:
            break

        result_offset += len(features)

    if not all_features:
        raise RuntimeError(
            "The official 2021 LSOA boundaries "
            "could not be downloaded."
        )

    gdf_all_boundaries = (
        gpd.GeoDataFrame.from_features(
            all_features,
            crs="EPSG:4326"
        )
    )

    gdf_all_boundaries = (
        gdf_all_boundaries
        .rename(
            columns={
                "LSOA21CD": "lsoa_code",
                "LSOA21NM": "lsoa_name"
            }
        )
    )

    study_lsoa_codes = set(
        df_imd[
            "lsoa_code"
        ].dropna()
    )

    gdf_boundaries = (
        gdf_all_boundaries[
            gdf_all_boundaries[
                "lsoa_code"
            ].isin(
                study_lsoa_codes
            )
        ]
        .copy()
    )

    gdf_boundaries.to_file(
        boundary_cache,
        driver="GeoJSON"
    )

    print(
        "Official 2021 boundary cache saved:",
        boundary_cache
    )

boundary_codes = set(
    gdf_boundaries[
        "lsoa_code"
    ].dropna()
)

imd_codes = set(
    df_imd[
        "lsoa_code"
    ].dropna()
)

unmatched_imd_codes = sorted(
    imd_codes
    - boundary_codes
)

boundary_match_rate = (
    (
        len(imd_codes)
        - len(unmatched_imd_codes)
    )
    / len(imd_codes)
    * 100
)

print(
    "Study-area IMD LSOAs:",
    len(imd_codes)
)

print(
    "LSOAs matched to official 2021 boundaries:",
    len(imd_codes)
    - len(unmatched_imd_codes)
)

print(
    f"Boundary match rate: "
    f"{boundary_match_rate:.2f}%"
)

if unmatched_imd_codes:
    pd.DataFrame(
        {
            "unmatched_lsoa_code":
                unmatched_imd_codes
        }
    ).to_csv(
        output_folder
        / "unmatched_imd_lsoa_codes.csv",
        index=False
    )

if boundary_match_rate < 99:
    raise ValueError(
        "Less than 99% of the study-area "
        "IMD LSOAs matched the official "
        "2021 boundaries."
    )

gdf = (
    gdf_boundaries
    .merge(
        df_imd,
        on="lsoa_code",
        how="inner",
        validate="one_to_one"
    )
)

gdf = gpd.GeoDataFrame(
    gdf,
    geometry="geometry",
    crs=gdf_boundaries.crs
)

print(
    "Final mapped study-area LSOAs:",
    gdf["lsoa_code"].nunique()
)

## 4. Parish- and Deanery-Level Analytical Measures

Church locations are spatially linked to LSOA-level deprivation data. Where a parish contains more than one mapped church site, median deprivation scores are used to create a single parish-level value.

For the exploratory dashboard, parish scores for the seven IMD domains are converted into relative percentiles. The aggregate need score is calculated as the equal-weighted mean of the income, employment, education, health, crime, barriers to housing and services, and living-environment percentiles. Overall IMD is retained separately as an official contextual deprivation measure.

Recorded provision is represented using successfully matched named APFR records. Parish provision percentiles are calculated only for parishes with matched project-record information; parishes without a matched named record remain separate rather than being assigned zero provision.

The alignment gap is calculated as:

**Alignment gap = aggregate need percentile − provision percentile**

An exploratory priority score is also calculated:

**Priority score = 0.70 × aggregate need + 0.30 × (100 − provision percentile)**

The weighting is an exploratory decision-support choice rather than a validated measure of service need. A separate review flag identifies comparatively high-need and lower-recorded-provision parishes, while high-need parishes with no matched named record are reported separately.

Deanery aggregation is used to provide a more stable comparison where parish reporting is uneven. The raw 605-record deanery analysis remains in the main Business Analysis notebook; this notebook focuses on the narrower matched-data spatial workflow.

### Sensitivity Checks

Two sensitivity checks were used to assess dependence on researcher-defined analytical choices.

First, deanery priority rankings were recalculated using 60/40, 70/30 and 80/20 weightings between aggregate need and comparatively low recorded provision.

Second, parish screening was compared using wider 70/30, main 75/25 and stricter 80/20 percentile cut-offs.

These checks assess the stability of the exploratory rankings and classifications rather than validating the measures as estimates of service need.

In [ ]:
dashboard_domains = {
    "Overall IMD": "IMDScore",
    "Income": "IncomeScore",
    "Employment": "EmpScore",
    "Education": "EduScore",
    "Health": "HealthScore",
    "Crime": "CrimeScore",
    "Housing and Services": "HousingScore",
    "Living Environment": "EnvScore"
}

domain_slugs = {
    "Overall IMD": "imd",
    "Income": "income",
    "Employment": "employment",
    "Education": "education",
    "Health": "health",
    "Crime": "crime",
    "Housing and Services": "housing",
    "Living Environment": "environment"
}

church_points = gpd.GeoDataFrame(
    df_churches.copy(),
    geometry=gpd.points_from_xy(
        df_churches["longitude"],
        df_churches["latitude"]
    ),
    crs="EPSG:4326"
).to_crs(
    gdf.crs
)

join_columns = list(
    dict.fromkeys(
        list(
            dashboard_domains.values()
        )
        + [
            "lsoa_code",
            "LADnm",
            "geometry"
        ]
    )
)

church_deprivation = (
    gpd.sjoin(
        church_points,
        gdf[join_columns],
        how="left",
        predicate="intersects"
    )
    .drop(
        columns="index_right",
        errors="ignore"
    )
)

unmatched_church_sites = (
    church_deprivation[
        church_deprivation[
            "lsoa_code"
        ].isna()
    ][
        [
            "episcopal_area",
            "deanery",
            "parish",
            "church",
            "postcode",
            "longitude",
            "latitude"
        ]
    ]
    .drop_duplicates()
)

if not unmatched_church_sites.empty:
    unmatched_church_sites.to_csv(
        output_folder
        / "unmatched_church_sites_to_lsoa.csv",
        index=False
    )

church_deprivation_valid = (
    church_deprivation[
        church_deprivation[
            "lsoa_code"
        ].notna()
    ]
    .copy()
)

parish_need = (
    church_deprivation_valid
    .groupby(
        [
            "location_key",
            "episcopal_area",
            "deanery",
            "parish"
        ],
        dropna=False
    )
    .agg(
        church_count=(
            "church",
            "nunique"
        ),
        longitude=(
            "longitude",
            "median"
        ),
        latitude=(
            "latitude",
            "median"
        ),
        lsoa_count=(
            "lsoa_code",
            "nunique"
        ),
        LADnm=(
            "LADnm",
            "first"
        ),
        IMDScore=(
            "IMDScore",
            "median"
        ),
        IncomeScore=(
            "IncomeScore",
            "median"
        ),
        EmpScore=(
            "EmpScore",
            "median"
        ),
        EduScore=(
            "EduScore",
            "median"
        ),
        HealthScore=(
            "HealthScore",
            "median"
        ),
        CrimeScore=(
            "CrimeScore",
            "median"
        ),
        HousingScore=(
            "HousingScore",
            "median"
        ),
        EnvScore=(
            "EnvScore",
            "median"
        )
    )
    .reset_index()
)

mapped_parish_keys = set(
    parish_need[
        "location_key"
    ]
)

analysis_projects = (
    matched_named_projects[
        matched_named_projects[
            "location_key"
        ].isin(
            mapped_parish_keys
        )
    ]
    .copy()
)

projects_without_imd = (
    matched_named_projects[
        ~matched_named_projects[
            "location_key"
        ].isin(
            mapped_parish_keys
        )
    ][
        [
            "episcopal_area",
            "deanery",
            "parish",
            "project"
        ]
    ]
    .drop_duplicates()
)

if not projects_without_imd.empty:
    projects_without_imd.to_csv(
        output_folder
        / "projects_without_imd_match.csv",
        index=False
    )

project_summary = (
    analysis_projects
    .groupby(
        "location_key",
        as_index=False
    )
    .agg(
        initiative_count=(
            "record_id",
            "nunique"
        ),
        people_supported=(
            "people_supported_for_analysis",
            sum_or_missing
        ),
        people_supported_records=(
            "people_supported_for_analysis",
            "count"
        ),
        volunteers=(
            "num_sa_volunteers",
            sum_or_missing
        ),
        volunteer_records=(
            "num_sa_volunteers",
            "count"
        ),
        project_names=(
            "project",
            combine_unique_text
        ),
        project_categories=(
            "category",
            combine_unique_text
        )
    )
)

category_exploded = (
    analysis_projects[
        [
            "location_key",
            "record_id",
            "category_tokens"
        ]
    ]
    .explode(
        "category_tokens"
    )
)

category_exploded = (
    category_exploded[
        category_exploded[
            "category_tokens"
        ].notna()
        & (
            category_exploded[
                "category_tokens"
            ] != ""
        )
    ]
)

category_summary = (
    category_exploded
    .groupby(
        "location_key",
        as_index=False
    )
    .agg(
        category_count=(
            "category_tokens",
            "nunique"
        )
    )
)

project_summary = (
    project_summary
    .merge(
        category_summary,
        on="location_key",
        how="left",
        validate="one_to_one"
    )
)

parish_dashboard = (
    parish_need
    .merge(
        project_summary,
        on="location_key",
        how="left",
        validate="one_to_one"
    )
)

parish_dashboard[
    "has_matched_named_project"
] = (
    parish_dashboard[
        "initiative_count"
    ].notna()
)

percentile_columns = []

for (
    domain_name,
    score_column
) in dashboard_domains.items():
    slug = domain_slugs[
        domain_name
    ]

    percentile_column = (
        f"{slug}_need_percentile"
    )

    parish_dashboard[
        percentile_column
    ] = (
        parish_dashboard[
            score_column
        ]
        .rank(
            method="average",
            pct=True
        )
        .mul(100)
    )

    percentile_columns.append(
        percentile_column
    )

subdomain_percentiles = [
    column
    for column in percentile_columns
    if column != "imd_need_percentile"
]

parish_dashboard[
    "exploratory_aggregate_need_score"
] = (
    parish_dashboard[
        subdomain_percentiles
    ]
    .mean(
        axis=1
    )
)

has_provision_data = (
    parish_dashboard[
        "has_matched_named_project"
    ]
)

parish_dashboard.loc[
    has_provision_data,
    "provision_percentile"
] = (
    parish_dashboard.loc[
        has_provision_data,
        "initiative_count"
    ]
    .rank(
        method="average",
        pct=True
    )
    .mul(100)
)

parish_dashboard[
    "alignment_gap"
] = (
    parish_dashboard[
        "exploratory_aggregate_need_score"
    ]
    - parish_dashboard[
        "provision_percentile"
    ]
)

parish_dashboard[
    "priority_score"
] = np.nan

parish_dashboard.loc[
    has_provision_data,
    "priority_score"
] = (
    0.70
    * parish_dashboard.loc[
        has_provision_data,
        "exploratory_aggregate_need_score"
    ]
    + 0.30
    * (
        100
        - parish_dashboard.loc[
            has_provision_data,
            "provision_percentile"
        ]
    )
)

parish_dashboard[
    "high_need_low_recorded_provision"
] = (
    has_provision_data
    & (
        parish_dashboard[
            "exploratory_aggregate_need_score"
        ] >= 75
    )
    & (
        parish_dashboard[
            "provision_percentile"
        ] <= 25
    )
)

parish_dashboard[
    "high_need_no_matched_project"
] = (
    ~has_provision_data
    & (
        parish_dashboard[
            "exploratory_aggregate_need_score"
        ] >= 75
    )
)

deanery_need = (
    parish_dashboard
    .groupby(
        [
            "episcopal_area",
            "deanery"
        ],
        dropna=False
    )
    .agg(
        parish_count=(
            "parish",
            "nunique"
        ),
        church_count=(
            "church_count",
            "sum"
        ),
        IMDScore=(
            "IMDScore",
            "median"
        ),
        IncomeScore=(
            "IncomeScore",
            "median"
        ),
        EmpScore=(
            "EmpScore",
            "median"
        ),
        EduScore=(
            "EduScore",
            "median"
        ),
        HealthScore=(
            "HealthScore",
            "median"
        ),
        CrimeScore=(
            "CrimeScore",
            "median"
        ),
        HousingScore=(
            "HousingScore",
            "median"
        ),
        EnvScore=(
            "EnvScore",
            "median"
        )
    )
    .reset_index()
)

deanery_provision = (
    parish_dashboard
    .groupby(
        [
            "episcopal_area",
            "deanery"
        ],
        dropna=False
    )
    .agg(
        initiative_count=(
            "initiative_count",
            sum_or_missing
        ),
        parishes_with_named_project_record=(
            "has_matched_named_project",
            "sum"
        ),
        people_supported=(
            "people_supported",
            sum_or_missing
        ),
        volunteers=(
            "volunteers",
            sum_or_missing
        )
    )
    .reset_index()
)

deanery_dashboard = (
    deanery_need
    .merge(
        deanery_provision,
        on=[
            "episcopal_area",
            "deanery"
        ],
        how="left",
        validate="one_to_one"
    )
)

deanery_dashboard[
    "project_record_coverage_pct"
] = (
    deanery_dashboard[
        "parishes_with_named_project_record"
    ]
    / deanery_dashboard[
        "parish_count"
    ]
    * 100
)

deanery_dashboard[
    "initiatives_per_project_record_parish"
] = (
    deanery_dashboard[
        "initiative_count"
    ]
    / deanery_dashboard[
        "parishes_with_named_project_record"
    ].replace(
        0,
        np.nan
    )
)

deanery_percentile_columns = []

for (
    domain_name,
    score_column
) in dashboard_domains.items():
    slug = domain_slugs[
        domain_name
    ]

    percentile_column = (
        f"{slug}_need_percentile"
    )

    deanery_dashboard[
        percentile_column
    ] = (
        deanery_dashboard[
            score_column
        ]
        .rank(
            method="average",
            pct=True
        )
        .mul(100)
    )

    deanery_percentile_columns.append(
        percentile_column
    )

deanery_subdomain_percentiles = [
    column
    for column in deanery_percentile_columns
    if column != "imd_need_percentile"
]

deanery_dashboard[
    "exploratory_aggregate_need_score"
] = (
    deanery_dashboard[
        deanery_subdomain_percentiles
    ]
    .mean(
        axis=1
    )
)

has_deanery_provision = (
    deanery_dashboard[
        "initiatives_per_project_record_parish"
    ].notna()
)

deanery_dashboard.loc[
    has_deanery_provision,
    "provision_percentile"
] = (
    deanery_dashboard.loc[
        has_deanery_provision,
        "initiatives_per_project_record_parish"
    ]
    .rank(
        method="average",
        pct=True
    )
    .mul(100)
)

deanery_dashboard[
    "alignment_gap"
] = (
    deanery_dashboard[
        "exploratory_aggregate_need_score"
    ]
    - deanery_dashboard[
        "provision_percentile"
    ]
)

deanery_dashboard[
    "priority_score"
] = np.nan

deanery_dashboard.loc[
    has_deanery_provision,
    "priority_score"
] = (
    0.70
    * deanery_dashboard.loc[
        has_deanery_provision,
        "exploratory_aggregate_need_score"
    ]
    + 0.30
    * (
        100
        - deanery_dashboard.loc[
            has_deanery_provision,
            "provision_percentile"
        ]
    )
)

deanery_dashboard["priority_score_60_40"] = np.nan
deanery_dashboard["priority_score_70_30"] = np.nan
deanery_dashboard["priority_score_80_20"] = np.nan

deanery_dashboard.loc[
    has_deanery_provision,
    "priority_score_60_40"
] = (
    0.60
    * deanery_dashboard.loc[
        has_deanery_provision,
        "exploratory_aggregate_need_score"
    ]
    + 0.40
    * (
        100
        - deanery_dashboard.loc[
            has_deanery_provision,
            "provision_percentile"
        ]
    )
)

deanery_dashboard.loc[
    has_deanery_provision,
    "priority_score_70_30"
] = (
    0.70
    * deanery_dashboard.loc[
        has_deanery_provision,
        "exploratory_aggregate_need_score"
    ]
    + 0.30
    * (
        100
        - deanery_dashboard.loc[
            has_deanery_provision,
            "provision_percentile"
        ]
    )
)

deanery_dashboard.loc[
    has_deanery_provision,
    "priority_score_80_20"
] = (
    0.80
    * deanery_dashboard.loc[
        has_deanery_provision,
        "exploratory_aggregate_need_score"
    ]
    + 0.20
    * (
        100
        - deanery_dashboard.loc[
            has_deanery_provision,
            "provision_percentile"
        ]
    )
)

sensitivity_top_deaneries = pd.DataFrame({
    "60/40": (
        deanery_dashboard
        .sort_values("priority_score_60_40", ascending=False)
        ["deanery"]
        .head(6)
        .reset_index(drop=True)
    ),
    "70/30": (
        deanery_dashboard
        .sort_values("priority_score_70_30", ascending=False)
        ["deanery"]
        .head(6)
        .reset_index(drop=True)
    ),
    "80/20": (
        deanery_dashboard
        .sort_values("priority_score_80_20", ascending=False)
        ["deanery"]
        .head(6)
        .reset_index(drop=True)
    )
})

display(sensitivity_top_deaneries)
parish_dashboard.to_csv(
    output_folder
    / "parish_dashboard_2025.csv",
    index=False
)

deanery_dashboard.to_csv(
    output_folder
    / "deanery_dashboard_2025.csv",
    index=False
)

matched_project_total = (
    analysis_projects[
        "record_id"
    ].nunique()
)

dashboard_project_total = (
    parish_dashboard[
        "initiative_count"
    ].sum(
        min_count=1
    )
)

if int(
    matched_project_total
) != int(
    dashboard_project_total
):
    raise ValueError(
        "The matched project total does not "
        "equal the parish project-count total."
    )


adjusted_deanery_correlation = (
    deanery_dashboard[
        [
            "exploratory_aggregate_need_score",
            "initiatives_per_project_record_parish"
        ]
    ]
    .corr(
        method="spearman"
    )
    .iloc[
        0,
        1
    ]
)

print(
    "Mapped parishes:",
    parish_dashboard[
        "parish"
    ].nunique()
)

print(
    "Matched named project records:",
    matched_project_total
)

print(
    "Parishes with at least one matched project:",
    int(
        parish_dashboard[
            "has_matched_named_project"
        ].sum()
    )
)


print(
    "Reporting-adjusted deanery correlation:",
    round(
        adjusted_deanery_correlation,
        3
    )
)

In [ ]:

threshold_specs = [
    ("70/30", 70, 30),
    ("75/25", 75, 25),
    ("80/20", 80, 20)
]

sensitivity_results = []
flagged_sets = {}

for label, need_threshold, provision_threshold in threshold_specs:

    mask = (
        parish_dashboard["has_matched_named_project"]
        & (
            parish_dashboard[
                "exploratory_aggregate_need_score"
            ] >= need_threshold
        )
        & (
            parish_dashboard[
                "provision_percentile"
            ] <= provision_threshold
        )
    )

    flagged = (
        parish_dashboard.loc[
            mask,
            [
                "parish",
                "deanery",
                "episcopal_area",
                "exploratory_aggregate_need_score",
                "provision_percentile",
                "alignment_gap",
                "priority_score"
            ]
        ]
        .sort_values(
            "exploratory_aggregate_need_score",
            ascending=False
        )
        .copy()
    )

    flagged_sets[label] = set(
        flagged["parish"].dropna()
    )

    sensitivity_results.append({
        "Threshold specification": label,
        "Need threshold": need_threshold,
        "Provision threshold": provision_threshold,
        "Number of flagged parishes": len(flagged),
        "Flagged parishes": ", ".join(
            flagged["parish"].dropna().tolist()
        )
    })

sensitivity_table = pd.DataFrame(
    sensitivity_results
)

print("Parish threshold sensitivity analysis")
display(sensitivity_table)


# Check how consistently each parish appears across the three specifications

all_flagged_parishes = sorted(
    set().union(*flagged_sets.values())
)

recurrence_results = []

for parish in all_flagged_parishes:

    specifications = [
        label
        for label, parishes in flagged_sets.items()
        if parish in parishes
    ]

    recurrence_results.append({
        "Parish": parish,
        "Number of specifications": len(specifications),
        "Appears under": ", ".join(specifications)
    })

recurrence_table = (
    pd.DataFrame(recurrence_results)
    .sort_values(
        ["Number of specifications", "Parish"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

print("\nParish recurrence across sensitivity specifications")
display(recurrence_table)

### Sensitivity Interpretation

The four leading deaneries remained North Kensington, Tower Hamlets, Haringey and Hackney across the alternative priority-weighting specifications.

Parish screening was more threshold-dependent: 11 parishes were identified under the wider 70/30 specification, four under the main 75/25 specification and one under the stricter 80/20 specification. Burnt Oak was identified under all three.

The results therefore support treating the screening outputs as exploratory and threshold-dependent.

## 5. Interactive Map Construction

A reusable Folium mapping function is used to create the interactive deprivation maps. Each map combines LSOA-level deprivation shading with church-location markers and, where appropriate, a heat layer representing matched named APFR records.

Each matched record included in a heat layer receives equal weight. Reported beneficiary and volunteer values are retained as contextual information but are not used to weight the heatmaps because these fields contain substantial missingness.

The heat layers therefore show the geographical concentration of recorded activity rather than project scale, effectiveness or total local provision.

In [ ]:
def project_matches_rule(
    row,
    category_any=None,
    text_regex=None
):
    category_any = (
        set(category_any)
        if category_any
        else set()
    )

    row_categories = set(
        row["category_tokens"]
    )

    category_match = bool(
        row_categories.intersection(
            category_any
        )
    )

    text_match = False

    if text_regex:
        text_match = bool(
            re.search(
                text_regex,
                row["project_search_text"],
                flags=re.IGNORECASE
            )
        )

    return category_match or text_match

def select_projects_for_rule(rule):
    if rule == "all":
        return (
            analysis_projects
            .drop_duplicates(subset=["record_id"])
            .copy()
        )

    if rule is None:
        return analysis_projects.iloc[0:0].copy()

    selected = analysis_projects[
        analysis_projects.apply(
            lambda row: project_matches_rule(
                row,
                category_any=rule.get(
                    "category_any"
                ),
                text_regex=rule.get(
                    "text_regex"
                )
            ),
            axis=1
        )
    ].copy()

    return selected.drop_duplicates(
        subset=[
            "record_id"
        ]
    )

def create_deprivation_map(
    score_field,
    output_filename,
    colormap_name,
    map_title,
    project_description,
    project_rule,
    no_project_note=None
):
    if score_field not in gdf.columns:
        raise KeyError(
            f"{score_field} was not found."
        )

    gdf_map = gdf.to_crs(
        epsg=4326
    )

    selected_projects = (
        select_projects_for_rule(
            project_rule
        )
    )

    project_count = (
        selected_projects[
            "record_id"
        ].nunique()
    )

    deprivation_group = folium.FeatureGroup(
        name="Deprivation layer",
        overlay=True,
        show=True
    )

    church_group = folium.FeatureGroup(
        name="Church locations",
        overlay=True,
        show=True
    )

    heat_group = None

    if project_count > 0:
        heat_group = folium.FeatureGroup(
            name="Matched project concentration",
            overlay=True,
            show=True
        )

    map_object = folium.Map(
        location=[
            51.65,
            -0.25
        ],
        tiles="cartodb positron",
        zoom_start=9,
        min_zoom=7,
        max_zoom=16,
        control_scale=True
    )

    Fullscreen(
        position="topleft",
        title="Open full screen",
        title_cancel="Exit full screen",
        force_separate_button=True
    ).add_to(
        map_object
    )

    score_values = pd.to_numeric(
        gdf_map[
            score_field
        ],
        errors="coerce"
    )

    colour_scale = getattr(
        linear,
        colormap_name
    ).scale(
        score_values.min(),
        score_values.max()
    )

    colour_scale.caption = (
        f"{map_title} – IMD 2025"
    )

    colour_scale.add_to(
        map_object
    )

    def style_function(feature):
        score = feature[
            "properties"
        ].get(
            score_field
        )

        if score is None or pd.isna(score):
            return {
                "fillColor": "#d9d9d9",
                "color": "#ffffff",
                "weight": 0.2,
                "fillOpacity": 0.15
            }

        return {
            "fillColor": colour_scale(score),
            "color": "#ffffff",
            "weight": 0.25,
            "fillOpacity": 0.60
        }

    folium.GeoJson(
        gdf_map,
        tooltip=folium.GeoJsonTooltip(
            fields=[
                "lsoa_code",
                "LADnm",
                score_field
            ],
            aliases=[
                "LSOA:",
                "Local authority:",
                f"{map_title}:"
            ],
            localize=True
        ),
        popup=folium.GeoJsonPopup(
            fields=[
                "lsoa_code",
                "LADnm",
                score_field
            ],
            aliases=[
                "LSOA:",
                "Local authority:",
                f"{map_title}:"
            ],
            localize=True
        ),
        style_function=style_function,
        highlight_function=lambda feature: {
            "weight": 1.2,
            "color": "#555555",
            "fillOpacity": 0.80
        }
    ).add_to(
        deprivation_group
    )

    for _, church_row in df_churches.iterrows():
        latitude = pd.to_numeric(
            church_row.get(
                "latitude"
            ),
            errors="coerce"
        )

        longitude = pd.to_numeric(
            church_row.get(
                "longitude"
            ),
            errors="coerce"
        )

        if pd.isna(latitude) or pd.isna(longitude):
            continue

        parish_key = church_row[
            "location_key"
        ]

        parish_name = safe_html(
            church_row.get(
                "parish"
            ),
            "Unknown parish"
        )

        church_name = safe_html(
            church_row.get(
                "church"
            ),
            ""
        )

        postcode = safe_html(
            church_row.get(
                "postcode"
            ),
            ""
        )

        district = safe_html(
            church_row.get(
                "admin_district"
            ),
            ""
        )

        projects_this_parish = (
            selected_projects[
                selected_projects[
                    "location_key"
                ] == parish_key
            ]
            .copy()
        )

        popup_html = f"""
        <div style="
            font-family: Arial, sans-serif;
            width: 360px;
        ">
            <h3 style="
                color: #b00020;
                margin-bottom: 8px;
            ">
                {parish_name}
            </h3>

            <p>
                <b>Church:</b>
                {church_name}
            </p>

            <p>
                <b>Location:</b>
                {postcode}
                {f"({district})" if district else ""}
            </p>
        """

        if projects_this_parish.empty:
            if project_count == 0 and no_project_note:
                popup_html += f"""
                <p>
                    {safe_html(no_project_note)}
                </p>
                """
            else:
                popup_html += """
                <p>
                    No matched named project relevant
                    to this deprivation domain was
                    identified in the available APFR
                    records.
                </p>
                """

        else:
            for _, project_row in (
                projects_this_parish
                .iterrows()
            ):
                project_name = safe_html(
                    project_row.get(
                        "project"
                    ),
                    "Unnamed project"
                )

                description = safe_html(
                    project_row.get(
                        "description"
                    ),
                    ""
                )

                people_supported = (
                    project_row.get(
                        "people_supported"
                    )
                )

                volunteers = (
                    project_row.get(
                        "num_sa_volunteers"
                    )
                )

                if pd.isna(
                    people_supported
                ):
                    people_text = (
                        "Not reported"
                    )
                else:
                    people_text = (
                        f"{people_supported:,.0f}"
                    )

                if pd.isna(
                    volunteers
                ):
                    volunteer_text = (
                        "Not reported"
                    )
                else:
                    volunteer_text = (
                        f"{volunteers:,.0f}"
                    )

                popup_html += f"""
                <hr>

                <p>
                    <b>Project:</b>
                    {project_name}
                </p>

                <p>
                    {description}
                </p>

                <p>
                    <b>
                        Reported people supported:
                    </b>
                    {people_text}
                </p>

                <p>
                    <b>
                        Reported volunteers:
                    </b>
                    {volunteer_text}
                </p>
                """

        popup_html += "</div>"

        if (
            cross_icon is not None
            and cross_icon.exists()
        ):
            marker_icon = folium.CustomIcon(
                str(
                    cross_icon
                ),
                icon_size=(
                    30,
                    30
                )
            )

        else:
            marker_icon = folium.Icon(
                color="darkred",
                icon="plus",
                prefix="fa"
            )

        folium.Marker(
            location=[
                latitude,
                longitude
            ],
            icon=marker_icon,
            popup=folium.Popup(
                popup_html,
                min_width=320,
                max_width=500
            ),
            lazy=True
        ).add_to(
            church_group
        )

    heatmap_gradients = {
        "IMDScore": {
            "0": "#fff5eb",
            "0.3": "#fdae6b",
            "0.6": "#e6550d",
            "1": "#a63603"
        },
        "IncomeScore": {
            "0": "#fff7fb",
            "0.3": "#f7b6d2",
            "0.6": "#de4d8a",
            "1": "#8e0152"
        },
        "EmpScore": {
            "0": "#eff3ff",
            "0.3": "#9ecae1",
            "0.6": "#4292c6",
            "1": "#084594"
        },
        "CrimeScore": {
            "0": "#fee5d9",
            "0.3": "#fcae91",
            "0.6": "#fb6a4a",
            "1": "#a50f15"
        },
        "HousingScore": {
            "0": "#fff7bc",
            "0.3": "#fee391",
            "0.6": "#fec44f",
            "1": "#cc4c02"
        },
        "EduScore": {
            "0": "#f2f0f7",
            "0.3": "#cbc9e2",
            "0.6": "#9e9ac8",
            "1": "#6a51a3"
        },
        "EnvScore": {
            "0": "#edf8e9",
            "0.3": "#bae4b3",
            "0.6": "#74c476",
            "1": "#238b45"
        },
        "HealthScore": {
            "0": "#fff5f0",
            "0.3": "#fcbba1",
            "0.6": "#fb6a4a",
            "1": "#cb181d"
        }
    }

    if project_count > 0:
        heat_data = (
            selected_projects[
                [
                    "latitude",
                    "longitude",
                    "record_id"
                ]
            ]
            .drop_duplicates(
                subset=[
                    "record_id"
                ]
            )
            .dropna(
                subset=[
                    "latitude",
                    "longitude"
                ]
            )
            .copy()
        )

        heat_data["weight"] = 1.0

        if project_count <= 10:
            heat_radius = 16
            heat_blur = 10
            heat_min_opacity = 0.35

        elif project_count <= 50:
            heat_radius = 24
            heat_blur = 16
            heat_min_opacity = 0.30

        else:
            heat_radius = 35
            heat_blur = 25
            heat_min_opacity = 0.25

        HeatMap(
            heat_data[
                [
                    "latitude",
                    "longitude",
                    "weight"
                ]
            ].values.tolist(),
            radius=heat_radius,
            blur=heat_blur,
            min_opacity=heat_min_opacity,
            gradient=(
                heatmap_gradients.get(
                    score_field
                )
            )
        ).add_to(
            heat_group
        )

    deprivation_group.add_to(
        map_object
    )

    if heat_group is not None:
        heat_group.add_to(
            map_object
        )

    church_group.add_to(
        map_object
    )

    folium.LayerControl(
        collapsed=True
    ).add_to(
        map_object
    )

    if project_count == 0:
        heat_explanation = (
            no_project_note
            or (
                "No matched named APFR project met "
                "the selection rule for this domain. "
                "The map therefore displays deprivation "
                "and church locations without a project "
                "heat layer."
            )
        )

    else:
        heat_explanation = (
            f"The heat layer contains {project_count} "
            "matched named APFR social-action records. "
            "Each record has equal weight, so darker "
            "or denser areas indicate a greater "
            "concentration of recorded activity."
        )

    information_panel = f"""
    <style>
    #map-information-panel {{
        position: fixed;
        bottom: 22px;
        left: 22px;
        width: 390px;
        max-height: 430px;
        overflow-y: auto;
        background: white;
        border-left: 7px solid #b00020;
        border-radius: 10px;
        padding: 20px;
        z-index: 9999;
        font-family: Arial, sans-serif;
        font-size: 13px;
        line-height: 1.5;
        box-shadow:
            0 5px 18px
            rgba(0, 0, 0, 0.22);
    }}

    #map-information-close {{
        position: absolute;
        top: 8px;
        right: 9px;
        width: 29px;
        height: 29px;
        border: none;
        border-radius: 50%;
        background: #b00020;
        color: white;
        font-size: 20px;
        font-weight: bold;
        cursor: pointer;
    }}

    #map-information-open {{
        display: none;
        position: fixed;
        bottom: 22px;
        left: 22px;
        width: 38px;
        height: 38px;
        border: none;
        border-radius: 50%;
        background: #b00020;
        color: white;
        font-size: 18px;
        font-weight: bold;
        cursor: pointer;
        z-index: 9999;
    }}
    </style>

    <div id="map-information-panel">
        <button
            id="map-information-close"
            onclick="closeMapInformation()"
        >
            ×
        </button>

        <div style="
            font-size: 18px;
            font-weight: bold;
            margin-bottom: 12px;
            padding-right: 36px;
        ">
            {map_title}
        </div>

        <b>Shaded areas:</b>
        LSOA-level IMD 2025 scores.
        Darker shading represents a higher
        deprivation score.

        <br><br>

        <b>Cross markers:</b>
        Geocoded church sites. Click a cross
        to view the parish and relevant matched
        project records.

        <br><br>

        <b>Project selection:</b>
        {project_description}.

        <br><br>

        <b>Heat layer:</b>
        {heat_explanation}

        <br><br>

        <b>Important limitation:</b>
        The map represents matched recorded APFR
        activity only. It does not measure all
        local activity, service quality, project
        effectiveness or total impact.

        <br><br>

        <i>
            Sources: English Indices of
            Deprivation 2025, official 2021
            LSOA boundaries and APFR 2025.
        </i>
    </div>

    <button
        id="map-information-open"
        onclick="openMapInformation()"
    >
        i
    </button>

    <script>
    function closeMapInformation() {{
        document.getElementById(
            "map-information-panel"
        ).style.display = "none";

        document.getElementById(
            "map-information-open"
        ).style.display = "block";
    }}

    function openMapInformation() {{
        document.getElementById(
            "map-information-panel"
        ).style.display = "block";

        document.getElementById(
            "map-information-open"
        ).style.display = "none";
    }}
    </script>
    """

    map_object.get_root().html.add_child(
        folium.Element(
            information_panel
        )
    )

    minx, miny, maxx, maxy = (
        gdf_map.total_bounds
    )

    map_object.fit_bounds(
        [
            [
                miny,
                minx
            ],
            [
                maxy,
                maxx
            ]
        ],
        padding=(
            25,
            25
        )
    )

    output_path = (
        output_folder
        / output_filename
    )

    map_object.save(
        str(
            output_path
        )
    )

    print(
        f"{map_title}: "
        f"{project_count} matched records"
    )

    return {
        "title": map_title,
        "filename": output_filename,
        "project_count": project_count
    }

## 6. Deprivation Map Suite

Eight interactive views are generated: overall IMD and each of the seven IMD domains.

The overall IMD map displays all successfully matched named APFR social-action records, providing the broadest spatial comparison between recorded activity and deprivation.

The individual domain maps use transparent category and keyword rules to identify records with an apparent thematic relationship to each deprivation domain. These selections are exploratory and do not imply that the selected projects directly address, cause or reduce the corresponding form of deprivation.

No explicitly employment-related matched APFR records were identified under the selected rule. The Employment Deprivation map therefore retains the deprivation and church-location layers without a project heat layer. This should not be interpreted as evidence that employment-related support is absent.

In [ ]:
map_configurations = [
    {
        "score_field":
            "IMDScore",
        "output_filename":
            "map_2025_imd.html",
        "colormap_name":
            "YlOrBr_09",
        "map_title":
            "Index of Multiple Deprivation",
        "project_description":
            "all matched named APFR social-action records",
        "project_rule":
            "all",
        "no_project_note":
            None
    },
    {
        "score_field":
            "IncomeScore",
        "output_filename":
            "map_2025_income.html",
        "colormap_name":
            "RdPu_09",
        "map_title":
            "Income Deprivation",
        "project_description":
            "food, poverty, financial-support, clothing and homelessness projects",
        "project_rule": {
            "category_any": {
                "food",
                "poverty_financial",
                "clothing_essentials",
                "homelessness",
                "fundraising_donations"
            }
        },
        "no_project_note":
            None
    },
    {
        "score_field":
            "EmpScore",
        "output_filename":
            "map_2025_employment.html",
        "colormap_name":
            "PuBu_09",
        "map_title":
            "Employment Deprivation",
        "project_description":
            "the employment-deprivation indicator is displayed without a project overlay because no explicitly employment-related matched APFR records were identified",
        "project_rule":
            None,
        "no_project_note":
            (
                "No explicitly employment-related matched "
                "APFR social-action records were identified. "
                "The employment-deprivation layer and church "
                "locations are therefore shown without a "
                "project heat layer. This does not demonstrate "
                "that employment support is absent, because "
                "relevant activity may not have been separately "
                "categorised or described in the available data."
            )
    },
    {
        "score_field":
            "CrimeScore",
        "output_filename":
            "map_2025_crime.html",
        "colormap_name":
            "Reds_09",
        "map_title":
            "Crime Deprivation",
        "project_description":
            "crime, safety, victim-support, violence, abuse, offending and addiction-related projects",
        "project_rule": {
            "category_any": {
                "addiction",
                "org_aa"
            },
            "text_regex":
                r"\b(crime|criminal|safety|victim|victims|violence|violent|abuse|domestic abuse|domestic violence|offending|offender|offenders|prison|prisoner|prisoners|rehabilitation)\b"
        },
        "no_project_note":
            None
    },
    {
        "score_field":
            "HousingScore",
        "output_filename":
            "map_2025_housing.html",
        "colormap_name":
            "YlOrBr_09",
        "map_title":
            "Barriers to Housing and Services",
        "project_description":
            "housing, homelessness, shelter, migrant, refugee and asylum-related projects",
        "project_rule": {
            "category_any": {
                "homelessness",
                "migrants_refugees"
            },
            "text_regex":
                r"\b(housing|homeless|homelessness|shelter|temporary accommodation|refugee|refugees|asylum|asylum seeker|asylum seekers|migrant|migrants|immigration)\b"
        },
        "no_project_note":
            None
    },
    {
        "score_field":
            "EduScore",
        "output_filename":
            "map_2025_education.html",
        "colormap_name":
            "Purples_09",
        "map_title":
            "Education, Skills and Training",
        "project_description":
            "children, families, youth, school, education, homework, tuition and training projects",
        "project_rule": {
            "category_any": {
                "families_children"
            },
            "text_regex":
                r"\b(school|schools|education|educational|homework|tuition|tutoring|training|skills|learning|literacy|student|students|youth|young people|children)\b"
        },
        "no_project_note":
            None
    },
    {
        "score_field":
            "EnvScore",
        "output_filename":
            "map_2025_environment.html",
        "colormap_name":
            "BuGn_09",
        "map_title":
            "Living Environment Deprivation",
        "project_description":
            "environment, recycling, gardening, climate and neighbourhood-improvement projects",
        "project_rule": {
            "category_any": {
                "environment"
            },
            "text_regex":
                r"\b(environment|environmental|recycling|recycle|garden|gardening|climate|green project|green space|neighbourhood improvement|community clean|litter|sustainability|sustainable)\b"
        },
        "no_project_note":
            None
    },
    {
        "score_field":
            "HealthScore",
        "output_filename":
            "map_2025_health.html",
        "colormap_name":
            "OrRd_09",
        "map_title":
            "Health Deprivation and Disability",
        "project_description":
            "health, mental-health, disability, dementia, elderly, wellbeing and addiction-related projects",
        "project_rule": {
            "category_any": {
                "elderly",
                "addiction",
                "org_aa"
            },
            "text_regex":
                r"\b(health|mental health|mental wellbeing|wellbeing|disability|disabled|dementia|elderly|older people|older person|addiction|alcoholics anonymous|alcohol support|drug support|medical|hospital|care home)\b"
        },
        "no_project_note":
            None
    }
]

map_results = []

for configuration in map_configurations:
    result = create_deprivation_map(
        score_field=(
            configuration[
                "score_field"
            ]
        ),
        output_filename=(
            configuration[
                "output_filename"
            ]
        ),
        colormap_name=(
            configuration[
                "colormap_name"
            ]
        ),
        map_title=(
            configuration[
                "map_title"
            ]
        ),
        project_description=(
            configuration[
                "project_description"
            ]
        ),
        project_rule=(
            configuration[
                "project_rule"
            ]
        ),
        no_project_note=(
            configuration[
                "no_project_note"
            ]
        )
    )

    map_results.append(
        result
    )

map_count_table = pd.DataFrame(
    map_results
)

map_count_table.to_csv(
    output_folder
    / "map_project_counts_2025.csv",
    index=False
)

display(
    map_count_table
)

## 7. Dashboard Visualisations

The analytical dashboard converts the parish- and deanery-level measures into a set of exploratory visualisations.

The charts examine aggregate deprivation, relationships between deprivation measures, recorded provision, reporting-adjusted deanery patterns, alignment gaps and potential priority areas.

These outputs are descriptive rather than causal. They are intended to identify geographical patterns and areas that may warrant further investigation alongside local knowledge and checks of reporting completeness.

In [ ]:
caritas_red = "#b00020"
caritas_light = "#e8a6b1"

top_need = (
    parish_dashboard
    .dropna(
        subset=[
            "exploratory_aggregate_need_score"
        ]
    )
    .nlargest(
        15,
        "exploratory_aggregate_need_score"
    )
    .sort_values(
        "exploratory_aggregate_need_score"
    )
)

fig, ax = plt.subplots(
    figsize=(
        11,
        7
    )
)

ax.barh(
    top_need["parish"],
    top_need[
        "exploratory_aggregate_need_score"
    ],
    color=caritas_red
)

ax.set_title(
    "Top 15 Parishes by Exploratory "
    "Aggregate Need Score",
    fontsize=16,
    fontweight="bold",
    pad=15
)

ax.set_xlabel(
    "Equal-weighted mean percentile "
    "across seven deprivation domains"
)

ax.set_ylabel("")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

for index, value in enumerate(
    top_need[
        "exploratory_aggregate_need_score"
    ]
):
    ax.text(
        value + 0.5,
        index,
        f"{value:.1f}",
        va="center",
        fontsize=9
    )

plt.tight_layout()

plt.savefig(
    output_folder
    / "dashboard_top_need.png",
    dpi=180,
    bbox_inches="tight"
)

plt.close()

correlation_data = (
    parish_dashboard[
        list(
            dashboard_domains.values()
        )
    ]
    .corr(
        method="spearman"
    )
)

correlation_data.index = list(
    dashboard_domains.keys()
)

correlation_data.columns = list(
    dashboard_domains.keys()
)

fig, ax = plt.subplots(
    figsize=(
        11,
        9
    )
)

image = ax.imshow(
    correlation_data.values,
    vmin=-1,
    vmax=1,
    cmap="coolwarm"
)

ax.set_xticks(
    range(
        len(
            correlation_data.columns
        )
    )
)

ax.set_xticklabels(
    correlation_data.columns,
    rotation=45,
    ha="right"
)

ax.set_yticks(
    range(
        len(
            correlation_data.index
        )
    )
)

ax.set_yticklabels(
    correlation_data.index
)

for row in range(
    len(
        correlation_data.index
    )
):
    for column in range(
        len(
            correlation_data.columns
        )
    ):
        ax.text(
            column,
            row,
            f"{correlation_data.iloc[row, column]:.2f}",
            ha="center",
            va="center",
            fontsize=9
        )

ax.set_title(
    "Correlation Between "
    "Deprivation Measures",
    fontsize=16,
    fontweight="bold",
    pad=15
)

colour_bar = fig.colorbar(
    image,
    ax=ax,
    shrink=0.8
)

colour_bar.set_label(
    "Spearman correlation"
)

plt.tight_layout()

plt.savefig(
    output_folder
    / "dashboard_correlations.png",
    dpi=180,
    bbox_inches="tight"
)

plt.close()

parish_scatter_data = (
    parish_dashboard
    .dropna(
        subset=[
            "exploratory_aggregate_need_score",
            "initiative_count",
            "alignment_gap"
        ]
    )
    .copy()
)

scatter_sizes = (
    35
    + parish_scatter_data[
        "church_count"
    ]
    .fillna(1)
    .clip(
        lower=1
    )
    * 25
)

fig, ax = plt.subplots(
    figsize=(
        11,
        7
    )
)

ax.scatter(
    parish_scatter_data[
        "exploratory_aggregate_need_score"
    ],
    parish_scatter_data[
        "initiative_count"
    ],
    s=scatter_sizes,
    color=caritas_red,
    alpha=0.60,
    edgecolors="white",
    linewidth=0.8
)

ax.axvline(
    parish_scatter_data[
        "exploratory_aggregate_need_score"
    ].median(),
    linestyle="--",
    linewidth=1,
    color="grey"
)

ax.axhline(
    parish_scatter_data[
        "initiative_count"
    ].median(),
    linestyle="--",
    linewidth=1,
    color="grey"
)

largest_parish_gaps = (
    parish_scatter_data
    .nlargest(
        8,
        "alignment_gap"
    )
)

for _, row in largest_parish_gaps.iterrows():
    ax.annotate(
        row["parish"],
        (
            row[
                "exploratory_aggregate_need_score"
            ],
            row[
                "initiative_count"
            ]
        ),
        xytext=(
            5,
            5
        ),
        textcoords="offset points",
        fontsize=8
    )

ax.set_title(
    "Exploratory Parish Need Compared with "
    "Matched Named Project Records",
    fontsize=16,
    fontweight="bold",
    pad=15
)

ax.set_xlabel(
    "Exploratory aggregate need score"
)

ax.set_ylabel(
    "Matched named project records"
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

plt.savefig(
    output_folder
    / "dashboard_alignment.png",
    dpi=180,
    bbox_inches="tight"
)

plt.close()

deanery_scatter_data = (
    deanery_dashboard
    .dropna(
        subset=[
            "exploratory_aggregate_need_score",
            "initiatives_per_project_record_parish",
            "alignment_gap"
        ]
    )
    .copy()
)

fig, ax = plt.subplots(
    figsize=(
        11,
        7
    )
)

ax.scatter(
    deanery_scatter_data[
        "exploratory_aggregate_need_score"
    ],
    deanery_scatter_data[
        "initiatives_per_project_record_parish"
    ],
    s=95,
    color=caritas_red,
    alpha=0.70,
    edgecolors="white",
    linewidth=0.8
)

largest_deanery_gaps = (
    deanery_scatter_data
    .nlargest(
        8,
        "alignment_gap"
    )
)

for _, row in largest_deanery_gaps.iterrows():
    ax.annotate(
        row["deanery"],
        (
            row[
                "exploratory_aggregate_need_score"
            ],
            row[
                "initiatives_per_project_record_parish"
            ]
        ),
        xytext=(
            5,
            5
        ),
        textcoords="offset points",
        fontsize=8
    )

ax.text(
    0.975,
    0.955,
    f"Spearman correlation: {adjusted_deanery_correlation:.2f}",
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=9,
    bbox={
        "boxstyle": "round,pad=0.40",
        "facecolor": "white",
        "edgecolor": "#777777",
        "linewidth": 0.8,
        "alpha": 0.96
    },
    zorder=6
)

ax.set_title(
    "Deanery Need Compared with Reporting-Adjusted Recorded Provision",
    fontsize=16,
    fontweight="bold",
    pad=16
)

ax.set_xlabel(
    "Exploratory Aggregate Need Score",
    fontsize=11
)

ax.set_ylabel(
    "Matched Named Records per Parish with a Matched Record",
    fontsize=11
)

ax.grid(
    alpha=0.15,
    linewidth=0.6
)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color("#444444")
    spine.set_linewidth(0.9)

plt.tight_layout()

plt.savefig(
    output_folder / "dashboard_deanery_alignment.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
top_priority_parishes = (
    parish_dashboard
    .dropna(
        subset=[
            "priority_score"
        ]
    )
    .nlargest(
        15,
        "priority_score"
    )
    .sort_values(
        "priority_score"
    )
)

fig, ax = plt.subplots(
    figsize=(
        11,
        7
    )
)

ax.barh(
    top_priority_parishes[
        "parish"
    ],
    top_priority_parishes[
        "priority_score"
    ],
    color=caritas_red
)

ax.set_title(
    "Potential Priority Parishes "
    "with Matched Project Data",
    fontsize=16,
    fontweight="bold",
    pad=15
)

ax.set_xlabel(
    "Exploratory priority score"
)

ax.set_ylabel("")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

for index, value in enumerate(
    top_priority_parishes[
        "priority_score"
    ]
):
    ax.text(
        value + 0.5,
        index,
        f"{value:.1f}",
        va="center",
        fontsize=9
    )

plt.tight_layout()

plt.savefig(
    output_folder
    / "dashboard_priority.png",
    dpi=180,
    bbox_inches="tight"
)

plt.close()

top_priority_deaneries = (
    deanery_dashboard
    .dropna(
        subset=[
            "priority_score"
        ]
    )
    .nlargest(
        15,
        "priority_score"
    )
    .sort_values(
        "priority_score"
    )
)

fig, ax = plt.subplots(
    figsize=(
        11,
        7
    )
)

ax.barh(
    top_priority_deaneries[
        "deanery"
    ],
    top_priority_deaneries[
        "priority_score"
    ],
    color=caritas_red
)

ax.set_title(
    "Deanery-Level Potential Priorities",
    fontsize=16,
    fontweight="bold",
    pad=15
)

ax.set_xlabel(
    "Exploratory priority score"
)

ax.set_ylabel("")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

for index, value in enumerate(
    top_priority_deaneries[
        "priority_score"
    ]
):
    ax.text(
        value + 0.5,
        index,
        f"{value:.1f}",
        va="center",
        fontsize=9
    )

plt.tight_layout()

plt.savefig(
    output_folder
    / "dashboard_deaneries.png",
    dpi=180,
    bbox_inches="tight"
)

plt.close()

deanery_gap_data = (
    deanery_dashboard
    .dropna(subset=["alignment_gap"])
    .sort_values("alignment_gap")
    .copy()
)


colors = [
    caritas_red if value > 0
    else caritas_light if value < 0
    else "#BDBDBD"
    for value in deanery_gap_data["alignment_gap"]
]

fig, ax = plt.subplots(
    figsize=(11, 8)
)

bars = ax.barh(
    deanery_gap_data["deanery"],
    deanery_gap_data["alignment_gap"],
    color=colors,
    height=0.72
)


ax.axvline(
    0,
    color="#444444",
    linewidth=1
)


for bar, value in zip(
    bars,
    deanery_gap_data["alignment_gap"]
):
    ax.text(
        value + (1.2 if value >= 0 else -1.2),
        bar.get_y() + bar.get_height() / 2,
        f"{value:+.1f}",
        va="center",
        ha="left" if value >= 0 else "right",
        fontsize=9
    )

ax.set_title(
    "Deanery-Level Exploratory Alignment Gaps",
    fontsize=16,
    fontweight="bold",
    pad=15
)

ax.set_xlabel(
    "Alignment Gap: Need Percentile − Provision Percentile"
)

ax.set_ylabel("")


maximum_gap = deanery_gap_data["alignment_gap"].abs().max()

ax.set_xlim(
    deanery_gap_data["alignment_gap"].min() - maximum_gap * 0.08,
    deanery_gap_data["alignment_gap"].max() + maximum_gap * 0.10
)


for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color("#444444")
    spine.set_linewidth(0.8)

plt.figtext(
    0.5,
    0.01,
    "Positive values indicate a higher need percentile than provision percentile. "
    "These gaps are exploratory review indicators.",
    ha="center",
    fontsize=9
)

plt.tight_layout(
    rect=[0, 0.04, 1, 1]
)

plt.savefig(
    output_folder / "dashboard_deanery_gap.png",
    dpi=180,
    bbox_inches="tight"
)

plt.show()

print(
    "Dashboard charts created in:",
    output_folder
)

## 8. Dashboard Tables and HTML Output

Supporting tables present the values underlying the dashboard visualisations. These include parish priority scores, high-need parishes without matched named records, deanery summaries and the highest-scoring parishes across the deprivation measures.

Missing matched-project information remains missing rather than being converted to zero. This distinction is important because an absence of a matched APFR record may reflect incomplete reporting or matching rather than an absence of social-action activity.

The charts and tables are then combined into a single HTML analytical dashboard.

In [ ]:
priority_table = (
    parish_dashboard
    .dropna(
        subset=[
            "priority_score"
        ]
    )
    .nlargest(
        20,
        "priority_score"
    )[
        [
            "parish",
            "deanery",
            "episcopal_area",
            "exploratory_aggregate_need_score",
            "initiative_count",
            "provision_percentile",
            "alignment_gap",
            "priority_score"
        ]
    ]
    .rename(
        columns={
            "parish":
                "Parish",
            "deanery":
                "Deanery",
            "episcopal_area":
                "Episcopal area",
            "exploratory_aggregate_need_score":
                "Exploratory aggregate need",
            "initiative_count":
                "Matched named project records",
            "provision_percentile":
                "Provision percentile",
            "alignment_gap":
                "Alignment gap",
            "priority_score":
                "Priority score"
        }
    )
    .round(1)
)

no_project_table = (
    parish_dashboard[
        parish_dashboard[
            "high_need_no_matched_project"
        ]
    ][
        [
            "parish",
            "deanery",
            "episcopal_area",
            "exploratory_aggregate_need_score",
            "IMDScore"
        ]
    ]
    .sort_values(
        "exploratory_aggregate_need_score",
        ascending=False
    )
    .rename(
        columns={
            "parish":
                "Parish",
            "deanery":
                "Deanery",
            "episcopal_area":
                "Episcopal area",
            "exploratory_aggregate_need_score":
                "Exploratory aggregate need",
            "IMDScore":
                "Official IMD score"
        }
    )
    .round(1)
)

deanery_table = (
    deanery_dashboard
    .dropna(
        subset=[
            "priority_score"
        ]
    )
    .sort_values(
        "priority_score",
        ascending=False
    )[
        [
            "deanery",
            "episcopal_area",
            "parish_count",
            "project_record_coverage_pct",
            "exploratory_aggregate_need_score",
            "initiative_count",
            "initiatives_per_project_record_parish",
            "alignment_gap",
            "priority_score"
        ]
    ]
    .rename(
        columns={
            "deanery":
                "Deanery",
            "episcopal_area":
                "Episcopal area",
            "parish_count":
                "Mapped parishes",
            "project_record_coverage_pct":
                "Parishes with matched record %",
            "exploratory_aggregate_need_score":
                "Exploratory aggregate need",
            "initiative_count":
                "Matched named project records",
            "initiatives_per_project_record_parish":
                "Matched named records per parish with matched record",
            "alignment_gap":
                "Alignment gap",
            "priority_score":
                "Priority score"
        }
    )
    .round(1)
)

domain_tables = []

for domain_name, score_column in dashboard_domains.items():
    percentile_column = (
        f"{domain_slugs[domain_name]}"
        "_need_percentile"
    )

    domain_table = (
        parish_dashboard
        .dropna(
            subset=[
                score_column
            ]
        )
        .nlargest(
            10,
            score_column
        )[
            [
                "parish",
                "deanery",
                score_column,
                percentile_column,
                "initiative_count",
                "alignment_gap"
            ]
        ]
        .rename(
            columns={
                "parish":
                    "Parish",
                "deanery":
                    "Deanery",
                score_column:
                    "Score",
                percentile_column:
                    "Need percentile",
                "initiative_count":
                    "Matched records",
                "alignment_gap":
                    "Alignment gap"
            }
        )
        .round(1)
    )
    domain_table = domain_table.fillna("—")

    domain_tables.append(
        f"""
        <div class="table-card">
            <h3>{domain_name}</h3>
            {
                domain_table.to_html(
                    index=False,
                    classes="data-table",
                    border=0
                )
            }
        </div>
        """
    )

total_mapped_parishes = int(
    parish_dashboard[
        "parish"
    ].nunique()
)

parishes_with_matched_project = int(
    parish_dashboard[
        "has_matched_named_project"
    ].sum()
)

project_record_coverage = (
    parishes_with_matched_project
    / total_mapped_parishes
    * 100
)

high_need_low_provision = int(
    parish_dashboard[
        "high_need_low_recorded_provision"
    ].sum()
)

high_need_no_project = int(
    parish_dashboard[
        "high_need_no_matched_project"
    ].sum()
)

dashboard_html = f"""
<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<title>
Caritas Westminster Analytical Dashboard
</title>
<style>
body {{
    font-family: Arial, sans-serif;
    background: #f7f7f7;
    margin: 0;
    padding: 0;
    color: #222;
}}

.container {{
    max-width: 1220px;
    margin: 0 auto;
    padding: 45px 28px 70px;
}}

.header {{
    background: white;
    border-left: 8px solid #b00020;
    border-radius: 12px;
    padding: 30px;
    margin-bottom: 28px;
    box-shadow: 0 4px 14px rgba(0,0,0,0.08);
}}

.header h1 {{
    margin: 0 0 12px;
    font-size: 38px;
}}

.header p {{
    margin: 0;
    color: #555;
    line-height: 1.6;
}}

.metric-grid {{
    display: grid;
    grid-template-columns:
        repeat(auto-fit, minmax(190px, 1fr));
    gap: 16px;
    margin-bottom: 30px;
}}

.metric-card {{
    background: white;
    border-radius: 10px;
    padding: 22px;
    box-shadow: 0 3px 12px rgba(0,0,0,0.07);
}}

.metric-label {{
    color: #666;
    font-size: 13px;
    margin-bottom: 8px;
}}

.metric-value {{
    color: #b00020;
    font-size: 30px;
    font-weight: bold;
}}

.section {{
    background: white;
    border-radius: 12px;
    padding: 26px;
    margin-bottom: 26px;
    box-shadow: 0 3px 12px rgba(0,0,0,0.07);
}}

.section h2 {{
    margin-top: 0;
}}

.section-description,
.chart-explanation {{
    color: #555;
    line-height: 1.55;
    margin-bottom: 18px;
}}

.chart {{
    display: block;
    width: 100%;
    max-width: 1050px;
    margin: 0 auto 16px;
}}

.data-table {{
    width: 100%;
    border-collapse: collapse;
    font-size: 13px;
}}

.data-table th {{
    background: #b00020;
    color: white;
    text-align: left;
    padding: 10px;
}}

.data-table td {{
    padding: 9px 10px;
    border-bottom: 1px solid #e3e3e3;
}}

.data-table tr:nth-child(even) {{
    background: #fafafa;
}}

.table-grid {{
    display: grid;
    grid-template-columns:
        repeat(auto-fit, minmax(480px, 1fr));
    gap: 20px;
}}

.table-card {{
    overflow-x: auto;
}}

.table-card h3 {{
    margin-top: 0;
    color: #b00020;
}}

.note {{
    background: #fff3f5;
    border-left: 7px solid #b00020;
    padding: 20px;
    line-height: 1.55;
    border-radius: 8px;
}}

.back-link {{
    display: inline-block;
    margin-bottom: 22px;
    color: #b00020;
    font-weight: bold;
    text-decoration: none;
}}

@media (max-width: 700px) {{
    .container {{
        padding: 25px 14px 45px;
    }}

    .header h1 {{
        font-size: 30px;
    }}

    .table-grid {{
        grid-template-columns: 1fr;
    }}
}}
</style>
</head>
<body>
<div class="container">

<a
    class="back-link"
    href="index_2025.html"
>
    ← Back to map homepage
</a>

<div class="header">
    <h1>
        Caritas Westminster Analytical Dashboard
    </h1>

    <p>
        Exploratory comparison of IMD 2025
        deprivation measures with matched named
        APFR social-action project records.
        Missing project reporting remains missing
        and is not treated as zero.
    </p>
</div>

<div class="metric-grid">
    <div class="metric-card">
        <div class="metric-label">
            Mapped parishes
        </div>
        <div class="metric-value">
            {total_mapped_parishes}
        </div>
    </div>

    <div class="metric-card">
        <div class="metric-label">
            Matched named project records
        </div>
        <div class="metric-value">
            {matched_project_total}
        </div>
    </div>

    <div class="metric-card">
        <div class="metric-label">
            Parishes with matched records
        </div>
        <div class="metric-value">
            {parishes_with_matched_project}
        </div>
    </div>

    <div class="metric-card">
        <div class="metric-label">
            Project-record coverage
        </div>
        <div class="metric-value">
            {project_record_coverage:.1f}%
        </div>
    </div>

    <div class="metric-card">
        <div class="metric-label">
            High need and lower recorded provision
        </div>
        <div class="metric-value">
            {high_need_low_provision}
        </div>
    </div>

    <div class="metric-card">
        <div class="metric-label">
            High need with no matched project record
        </div>
        <div class="metric-value">
            {high_need_no_project}
        </div>
    </div>
</div>

<div class="section">
    <h2>
        Highest Exploratory Aggregate Need
    </h2>

    <img
        class="chart"
        src="dashboard_top_need.png"
        alt="Top parishes by exploratory aggregate need"
    >

    <div class="chart-explanation">
        The aggregate need measure is the
        equal-weighted mean of parish percentiles
        across the seven IMD domains. It is an
        exploratory comparison tool rather than
        an official deprivation index.
    </div>
</div>

<div class="section">
    <h2>
        Correlation Between Deprivation Measures
    </h2>

    <img
        class="chart"
        src="dashboard_correlations.png"
        alt="Correlation matrix of deprivation measures"
    >

    <div class="chart-explanation">
        The matrix shows Spearman correlations
        between the parish-level deprivation
        measures assigned from church locations.
    </div>
</div>

<div class="section">
    <h2>
        Parish Need and Recorded Provision
    </h2>

    <img
        class="chart"
        src="dashboard_alignment.png"
        alt="Parish need compared with matched project records"
    >

    <div class="chart-explanation">
        The chart compares exploratory aggregate
        need with the number of matched named APFR
        project records. It does not measure
        effectiveness or total local provision.
    </div>
</div>

<div class="section">
    <h2>
        Deanery Need and Reporting-Adjusted Provision
    </h2>

    <img
        class="chart"
        src="dashboard_deanery_alignment.png"
        alt="Deanery need compared with reporting-adjusted provision"
    >

    <div class="chart-explanation">
        Recorded provision is expressed as matched named APFR
records per parish with a matched record, reducing
the direct influence of uneven project-record coverage.
    </div>
</div>

<div class="section">
    <h2>
        Potential Priority Parishes
    </h2>

    <img
        class="chart"
        src="dashboard_priority.png"
        alt="Potential priority parishes"
    >

    <div class="chart-explanation">
        The exploratory priority score weights
        aggregate need at 70% and lower recorded
        provision at 30%. It identifies areas for
        further investigation, not confirmed
        under-provision.
    </div>
</div>

<div class="section">
    <h2>
        Deanery-Level Potential Priorities
    </h2>

    <img
        class="chart"
        src="dashboard_deaneries.png"
        alt="Deanery-level potential priorities"
    >

    <div class="chart-explanation">
        This chart applies the exploratory priority
        measure at deanery level. Deanery-level
        medians provide a more stable basis for
        interpretation where parish reporting is
        incomplete.
    </div>
</div>

<div class="section">
    <h2>
        Deanery Alignment Gap
    </h2>

    <img
        class="chart"
        src="dashboard_deanery_gap.png"
        alt="Deanery alignment gap"
    >

    <div class="chart-explanation">
        Positive values indicate a higher need
        percentile than provision percentile.
        These are descriptive mismatches requiring
        further review rather than evidence of
        poor performance or unmet need.
    </div>
</div>

<div class="section">
    <h2>
        Potential Priority Parishes
    </h2>

    <div class="section-description">
        Highest exploratory parish priority scores
        together with the underlying deprivation,
        provision and alignment measures.
    </div>

    {
        priority_table.to_html(
            index=False,
            classes="data-table",
            border=0
        )
    }
</div>

<div class="section">
    <h2>
        High-Need Parishes Without a Matched Project Record
    </h2>

    <div class="section-description">
        These parishes have high exploratory need
        scores but no matched named APFR project
        record. This may reflect missing or
        incomplete reporting and must not be
        interpreted as zero activity.
    </div>

    {
        no_project_table.to_html(
            index=False,
            classes="data-table",
            border=0
        )
    }
</div>

<div class="section">
    <h2>
        Deanery Summary
    </h2>

    <div class="section-description">
        Deanery-level deprivation, reporting
        coverage and recorded provision.
    </div>

    {
        deanery_table.to_html(
            index=False,
            classes="data-table",
            border=0
        )
    }
</div>

<div class="section">
    <h2>
        Top 10 Parishes by Each Deprivation Measure
    </h2>

    <div class="section-description">
        The ten highest-scoring mapped parishes for
        each deprivation measure, with matched
        project counts and alignment gaps where
        available.
    </div>

    <div class="table-grid">
        {''.join(domain_tables)}
    </div>
</div>

<div class="note">
    <b>Data limitation:</b>
    The dashboard uses recorded APFR activity.
    Missing records are kept as missing rather than
    being treated as zero. Parish-level results are
    therefore exploratory. Deanery-level medians
    provide a more stable basis for the main
    dissertation interpretation.
</div>

</div>
</body>
</html>
"""

dashboard_path = (
    output_folder
    / "dashboard_2025.html"
)

with open(
    dashboard_path,
    "w",
    encoding="utf-8"
) as file:
    file.write(
        dashboard_html
    )

print(
    "Dashboard created:",
    dashboard_path
)

## 9. Connected Interactive Homepage

A connected HTML homepage brings together the analytical dashboard and the eight interactive deprivation maps.

The interface allows users to move between the overall analytical findings and individual geographical views of deprivation, church locations and matched recorded activity.

The homepage is designed as a practical decision-support output for exploring patterns rather than as a definitive assessment of parish performance.

In [ ]:
map_card_metadata = {
    "map_2025_imd.html": {
        "class_name": "imd",
        "label": "Overall IMD",
        "description": (
            "Overall Index of Multiple Deprivation "
            "with matched broad social-action records."
        )
    },
    "map_2025_income.html": {
        "class_name": "income",
        "label": "Income",
        "description": (
            "Income deprivation with food, poverty, "
            "financial-support, clothing and "
            "homelessness records."
        )
    },
    "map_2025_employment.html": {
        "class_name": "employment",
        "label": "Employment",
        "description": (
            "Employment-deprivation shading and church "
            "locations. No project heat layer is shown "
            "because no explicitly employment-related "
            "matched APFR records were identified."
        )
    },
    "map_2025_crime.html": {
        "class_name": "crime",
        "label": "Crime",
        "description": (
            "Crime deprivation with matched safety, "
            "victim-support, offending and "
            "addiction-related records."
        )
    },
    "map_2025_housing.html": {
        "class_name": "housing",
        "label": "Housing and Services",
        "description": (
            "Barriers to housing and services with "
            "housing, homelessness, migrant, refugee "
            "and asylum-related records."
        )
    },
    "map_2025_education.html": {
        "class_name": "education",
        "label": "Education",
        "description": (
            "Education, skills and training deprivation "
            "with children, youth, school and "
            "training-related records."
        )
    },
    "map_2025_environment.html": {
        "class_name": "environment",
        "label": "Living Environment",
        "description": (
            "Living-environment deprivation with "
            "matched environmental and "
            "neighbourhood-improvement records."
        )
    },
    "map_2025_health.html": {
        "class_name": "health",
        "label": "Health",
        "description": (
            "Health deprivation and disability with "
            "matched health, mental-health, disability, "
            "elderly and addiction-related records."
        )
    }
}

map_cards = []

for result in map_results:
    metadata = map_card_metadata[
        result["filename"]
    ]

    if result["project_count"] == 0:
        count_text = (
            "0 matched records — deprivation layer only"
        )
    elif result["project_count"] == 1:
        count_text = (
            "1 matched APFR social-action record"
        )
    else:
        count_text = (
            f"{result['project_count']} matched "
            "APFR social-action records"
        )

    map_cards.append(
        f"""
        <a
            class="map-card {metadata['class_name']}"
            href="{result['filename']}"
        >
            <div class="map-label">
                {metadata['label']}
            </div>

            <div class="map-description">
                {metadata['description']}
            </div>

            <div class="map-count">
                {count_text}
            </div>
        </a>
        """
    )

map_cards_html = "\n".join(
    map_cards
)

index_html = f"""
<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<title>
Social Action and Deprivation Maps
</title>
<style>
body {{
    font-family: Arial, sans-serif;
    background: #f7f7f7;
    margin: 0;
    padding: 0;
    color: #222;
}}

.container {{
    max-width: 1200px;
    margin: 0 auto;
    padding: 55px 30px 70px;
}}

.header {{
    text-align: center;
    margin-bottom: 42px;
}}

h1 {{
    font-size: 44px;
    margin: 0 0 14px;
}}

.subtitle {{
    max-width: 850px;
    margin: 0 auto;
    color: #555;
    font-size: 18px;
    line-height: 1.6;
}}

.dashboard-card {{
    display: block;
    background: #b00020;
    color: white;
    text-decoration: none;
    padding: 30px;
    border-radius: 14px;
    margin-bottom: 38px;
    box-shadow: 0 5px 16px rgba(0,0,0,0.12);
    transition: 0.2s;
}}

.dashboard-card:hover {{
    transform: translateY(-3px);
    box-shadow: 0 9px 24px rgba(0,0,0,0.18);
}}

.dashboard-label {{
    font-size: 12px;
    letter-spacing: 1.5px;
    opacity: 0.85;
    margin-bottom: 8px;
}}

.dashboard-title {{
    font-size: 28px;
    font-weight: bold;
    margin-bottom: 10px;
}}

.dashboard-description {{
    line-height: 1.55;
    max-width: 850px;
}}

.section-heading {{
    margin: 0 0 20px;
    font-size: 25px;
}}

.map-grid {{
    display: grid;
    grid-template-columns:
        repeat(3, 1fr);
    gap: 20px;
}}

.map-card {{
    --accent: #b00020;
    display: block;
    background: white;
    border-top: 7px solid var(--accent);
    color: #222;
    text-decoration: none;
    padding: 24px;
    border-radius: 11px;
    box-shadow: 0 4px 14px rgba(0,0,0,0.08);
    transition: 0.2s;
}}

.map-card:hover {{
    transform: translateY(-3px);
    box-shadow: 0 8px 22px rgba(0,0,0,0.14);
}}

.map-label {{
    color: var(--accent);
    font-size: 21px;
    font-weight: bold;
    margin-bottom: 10px;
}}

.map-description {{
    color: #555;
    line-height: 1.5;
    min-height: 88px;
}}

.map-count {{
    margin-top: 16px;
    font-size: 13px;
    font-weight: bold;
    color: var(--accent);
}}

.imd {{
    --accent: #d95f02;
}}

.income {{
    --accent: #c2185b;
}}

.employment {{
    --accent: #2878b5;
}}

.crime {{
    --accent: #c62828;
}}

.housing {{
    --accent: #d89b00;
}}

.education {{
    --accent: #7354a3;
}}

.environment {{
    --accent: #388e5a;
}}

.health {{
    --accent: #e05a47;
}}

.footer {{
    margin-top: 45px;
    text-align: center;
    font-size: 13px;
    color: #777;
    line-height: 1.5;
}}

@media (max-width: 900px) {{
    .map-grid {{
        grid-template-columns:
            repeat(2, 1fr);
    }}
}}

@media (max-width: 600px) {{
    .map-grid {{
        grid-template-columns: 1fr;
    }}

    h1 {{
        font-size: 34px;
    }}
}}
</style>
</head>
<body>
<div class="container">

<div class="header">
    <h1>
        Social Action and Deprivation Maps
    </h1>

    <div class="subtitle">
        Interactive visualisation of matched named
        parish social-action project records and
        IMD 2025 deprivation indicators across the
        Diocese of Westminster.
    </div>
</div>

<a
    class="dashboard-card"
    href="dashboard_2025.html"
>
    <div class="dashboard-label">
        ANALYSIS AND INSIGHTS
    </div>

    <div class="dashboard-title">
        Analytical Dashboard
    </div>

    <div class="dashboard-description">
        Explore parish and deanery deprivation,
        matched recorded provision, project-record
        coverage, alignment gaps and exploratory
        priority areas for further review.
    </div>
</a>

<h2 class="section-heading">
    Deprivation Maps
</h2>

<div class="map-grid">
    {map_cards_html}
</div>

<div class="footer">
    Created using the English Indices of
    Deprivation 2025, official 2021 LSOA
    boundaries, geocoded church-location data
    and APFR 2025 project records.
    <br>
    Missing or unmatched reporting should not be
    interpreted as an absence of social-action
    activity.
</div>

</div>
</body>
</html>
"""

index_path = (
    output_folder
    / "index_2025.html"
)

with open(
    index_path,
    "w",
    encoding="utf-8"
) as file:
    file.write(
        index_html
    )

print(
    "Homepage created:",
    index_path
)

## 10. Validation and Final Checks

The final stage validates the main analytical populations and dashboard outputs before interpretation.

The checks confirm the number of raw APFR records, named project records, successfully matched spatial records, mapped parishes, project-record coverage and exploratory review flags.

The reporting-adjusted deanery association is also retained as a final validation output for this spatial workflow. The separate raw deanery association based on all 605 APFR records is calculated in the main Business Analysis notebook.

These checks ensure that the full APFR analysis and the narrower matched spatial analysis remain conceptually distinct and that missing reporting is not inadvertently treated as zero provision.

In [ ]:
validation_summary = pd.DataFrame(
    {
        "Measure": [
            "Raw APFR rows",
            "Named APFR project records",
            "Named projects matched to parish geography",
            "Named projects used in final analysis",
            "Mapped parishes",
            "Parishes with matched project data",
            "Project-record coverage percentage",
            "Employment-related matched records shown on map",
            "High-need / lower-recorded-provision parishes",
            "High-need parishes with no matched project",
            "Reporting-adjusted deanery correlation"
        ],
        "Value": [
            len(df_projects_raw),
            len(named_projects),
            len(matched_named_projects),
            analysis_projects["record_id"].nunique(),
            total_mapped_parishes,
            parishes_with_matched_project,
            round(project_record_coverage, 1),
            int(
                map_count_table.loc[
                    map_count_table["title"] == "Employment Deprivation",
                    "project_count"
                ].iloc[0]
            ),
            high_need_low_provision,
            high_need_no_project,
            round(adjusted_deanery_correlation, 3)
        ]
    }
)

In [ ]:
display(validation_summary)

validation_summary.to_csv(
    output_folder / "validation_summary_2025.csv",
    index=False
)

print(
    "Main homepage:",
    index_path
)

print(
    "Dashboard:",
    dashboard_path
)

print(
    "\nInteractive outputs created successfully."
    "\nOpen index_2025.html from the output folder in a web browser."
)